In [1]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

import MDAnalysis as mda
from MDAnalysis.analysis import distances
import networkx as nx

from polym_analysis import *
from gromacs_topology import *

/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/Bio/Application/__init__.py:40: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:
def random_amides(polym, num, seed=12345):
    '''Randomly select num of amide bonds to break'''

    sel = polym.universe.select_atoms('type n')
    n = len(sel)
    rng = np.random.default_rng(seed=seed)
    rand_idx = rng.choice(np.arange(0,n), size=num, replace=False)
    
    return sel[rand_idx]

In [12]:
path = '/Users/nate_schwindt/Projects/polymer_membrane/polyamide_XAS_MD/Na/NaCl_polymer/50pct_deprotonated/OPC3/placed_Na/'

xlink_c = 'c and bonded type n'
xlink_n = 'n'
term_n = 'nh'
cl_type = 'cl'
oh_type = 'oh'
ow_type = 'OW'
hw_type = 'HW'
cation_type = 'Na'
anion_type = 'Cl'

poly = PolymAnalysis(path+'prod.gro', frmt='GRO', tpr_file=path+'prod.tpr',
                    xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n,
                    cl_type=cl_type, oh_type=oh_type, ow_type=ow_type, hw_type=hw_type,
                    cation_type=cation_type, anion_type=anion_type)

Initializing polyamide membrane for analysis...
PA membrane consists of 716 MPD monomers, 444 TMC monomers, 6120 waters for a total of 36086 atoms
Finished loading PA membrane for analysis!



In [13]:
# determine waters to remove and positions to add Cl and OH
target_crosslink_pct = 0.8
radius_for_water_search = 6  # angstroms
rng_seed = 3

n_xlink = len(poly.universe.select_atoms('type n'))
n_possible = len(poly.universe.select_atoms('type n nh'))
n_to_break = n_xlink - int(target_crosslink_pct * n_possible)

poly.create_polymer_graphs()
n_components = [nx.number_connected_components(g) for g in poly.polymer_graphs]

print(f'Going from {100 * n_xlink / n_possible:.1f}% crosslinking to {100 * (n_xlink - n_to_break) / n_possible:.1f}% crosslinking by breaking {n_to_break} amide bonds')
                                  
print('Number of water molecules before breaking crosslinks:', len(poly.universe.select_atoms('resname SOL').residues))

# delete amide bonds
amide_Ns = random_amides(poly, n_to_break, seed=rng_seed)
amide_Cs = poly.universe.select_atoms('(type c) and (bonded group amide_Ns)', amide_Ns=amide_Ns)

to_delete = []

print(f'Trying to delete {n_to_break} waters and adding {n_to_break} Cl atoms')
Cl_positions = np.zeros((n_to_break, 3))
for i,n in enumerate(amide_Ns):
    my_waters = poly.universe.atoms.select_atoms(f'(type OW) and (sphzone {radius_for_water_search} index {n.index})')
    if len(my_waters) == 0:
        raise ValueError('No waters found near amide N atom index', n.index)
    
    # save the positions of the waters to be replaced
    Cl_positions[i] = my_waters[0].position

    # remove the water molecule
    to_delete.append(my_waters[0].residue)
    
# locate nearby waters to replace with OH
print(f'Trying to delete {n_to_break} waters and adding {n_to_break} O atoms and {n_to_break} H atoms')
O_positions = np.zeros((n_to_break, 3))
H_positions = np.zeros((int(n_to_break/2), 3))
for i,c in enumerate(amide_Cs):
    my_waters = poly.universe.atoms.select_atoms(f'(type OW) and (sphzone {radius_for_water_search} index {c.index})')
    if len(my_waters) == 0:
        raise ValueError('No waters found near amide C atom index', c.index)
    
    # save the positions of the waters to be replaced
    O_positions[i] = my_waters[0].position

    if i < int(n_to_break/2):
        H_positions[i] = my_waters[0].bonded_atoms[0].position  # for 50% deprotonated, we want to add O- for half and OH for half

    # remove the water molecule
    to_delete.append(my_waters[0].residue)

to_delete = list(set(to_delete))  # remove duplicates

# Now delete all waters at once
print(f'Actually deleting {len(to_delete)} water molecules')
for i,water_res in enumerate(to_delete):
    if i == 0:
        all_waters_to_delete = water_res.atoms
    else:
        all_waters_to_delete += water_res.atoms

to_delete_idx = all_waters_to_delete.indices
poly.universe.delete_bonds([b for b in all_waters_to_delete.bonds]) # also all bonds associated with deleted waters
poly.universe.atoms = poly.universe.atoms.subtract(all_waters_to_delete)

# reassign residue numbers after deleted waters
dims = poly.universe.dimensions # save original dimensions
n_residues = len(poly.atoms.residues)
resids = np.arange(1, n_residues+1)
poly.universe.add_TopologyAttr('resid', list(resids))

print('Number of water molecules after breaking crosslinks:', len(poly.universe.select_atoms('resname SOL').residues))

poly.universe.atoms.write('./removed_waters.gro')

Going from 93.2% crosslinking to 80.0% crosslinking by breaking 189 amide bonds
Number of water molecules before breaking crosslinks: 6120
Trying to delete 189 waters and adding 189 Cl atoms
Trying to delete 189 waters and adding 189 O atoms and 189 H atoms
Actually deleting 206 water molecules
Number of water molecules after breaking crosslinks: 5914


In [ ]:
# update the topology file
top = GromacsTopology('./PA_ions.top', verbose=False)
water_mol = [mol for mol in top.molecules if mol.name == 'SOL'][0]
water_mol.n_mols = len(poly.universe.select_atoms('resname SOL').residues) # all we have to do is update the number of water molecules

top.write('./removed_waters.top')



18 molecule types in the topology for a total of 6223 molecules
Finished writing topology to ./removed_waters.top!
